<a href="https://colab.research.google.com/github/kieron-s/Comp3610-Assignment3/blob/main/Comp3610_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pyspark -q

In [4]:
import time
import pandas as pd
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("COMP3610_Assignment3_NYC_Taxi")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("Warn")
print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")

Spark version: 4.0.2
Application name: COMP3610_Assignment3_NYC_Taxi


In [5]:
DATA_PATH = "/content/yellow_tripdata_2024-01.parquet"

spark_start = time.time()
df_raw = spark.read.parquet(DATA_PATH)
spark_row_count = df_raw.count()
spark_load_time = time.time() - spark_start

pandas_start = time.time()
df_pandas = pd.read_parquet(DATA_PATH)
pandas_load_time = time.time() - pandas_start

print(f"\nLoad Time Comparison")
print(f"Spark  load time : {spark_load_time:.2f}s  ({spark_row_count:,} rows)")
print(f"Pandas load time : {pandas_load_time:.2f}s  ({len(df_pandas):,} rows)")
print(f"\nInterpretation: Spark has higher overhead on a single machine due to JVM")
print(f"startup and task scheduling. However, Spark scales across clusters and handles")
print(f"datasets far exceeding available RAM, making it essential for big data workloads.")


Load Time Comparison
Spark  load time : 14.87s  (2,964,624 rows)
Pandas load time : 1.43s  (2,964,624 rows)

Interpretation: Spark has higher overhead on a single machine due to JVM
startup and task scheduling. However, Spark scales across clusters and handles
datasets far exceeding available RAM, making it essential for big data workloads.


In [6]:
print(f"DataFrame Schema")
df_raw.printSchema()
print(f"\nTotal Rows: {spark_row_count:,}")
print(f"\nPartition Information: {df_raw.rdd.getNumPartitions()}")

DataFrame Schema
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)


Total Rows: 2,964,624

Partition Information: 2


In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance"
]

initial_count = df_raw.count()
print(f"Initial Row Count: {initial_count:,}")

df_no_nulls = df_raw.dropna(subset = critical_cols)
after_null_drop = df_no_nulls.count()
print(f"After Null Drop: {after_null_drop:,} (removed {initial_count - after_null_drop:,}")

df_filtered = df_no_nulls.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 500) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

after_filter = df_filtered.count()
print(f"After Filter: {after_filter:,} (removed {after_null_drop - after_filter:,}")
print(f"Total rows removed: {initial_count - after_filter:,}")

Initial Row Count: 2,964,624
After Null Drop: 2,964,624 (removed 0
After Filter: 2,870,046 (removed 94,578
Total rows removed: 94,578


In [8]:
df_clean = df_filtered.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0)
    ).otherwise(F.lit(None).cast(DoubleType()))
).withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
).withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime")
).withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 0,
        (F.col("tip_amount") / F.col("fare_amount")) * 100
    ).otherwise(F.lit(0.0))
)

print("Derived Columns:")
df_clean.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount",
    "trip_duration_minutes", "trip_speed_mph",
    "pickup_hour", "pickup_day_of_week", "tip_percentage"
).show(5, truncate=False)

print(f"\nClean row count: {df_clean.count():,}")


Derived Columns:
+--------------------+-------------+-----------+---------------------+------------------+-----------+------------------+------------------+
|tpep_pickup_datetime|trip_distance|fare_amount|trip_duration_minutes|trip_speed_mph    |pickup_hour|pickup_day_of_week|tip_percentage    |
+--------------------+-------------+-----------+---------------------+------------------+-----------+------------------+------------------+
|2024-01-01 00:57:55 |1.72         |17.7       |19.8                 |5.212121212121212 |0          |2                 |0.0               |
|2024-01-01 00:03:00 |1.8          |10.0       |6.6                  |16.363636363636363|0          |2                 |37.5              |
|2024-01-01 00:17:06 |4.7          |23.3       |17.916666666666668   |15.739534883720932|0          |2                 |12.875536480686694|
|2024-01-01 00:36:38 |1.4          |10.0       |8.3                  |10.120481927710843|0          |2                 |20.0              |
|20

In [9]:
df_clean.createOrReplaceTempView("taxi_trips")
print("View 'taxi_trips' registered.")

View 'taxi_trips' registered.


In [15]:
print("Query 1: Top 10 Busiest Pickup Hours")
q1 = spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*)                        AS trip_count,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_percent
    FROM taxi_trips
    GROUP BY pickup_hour
    ORDER BY trip_count DESC
    LIMIT 10
""")
q1.show()

print("Interpretation: \n\nEvening hours around 3 p.m onwards have the majority of trip volume, this is because of after work commuting.")
print("\nTip percentages tend to hover around 20% with the highest being ~2% more at night")

Query 1: Top 10 Busiest Pickup Hours
+-----------+----------+--------+---------------+
|pickup_hour|trip_count|avg_fare|avg_tip_percent|
+-----------+----------+--------+---------------+
|         18|    206281|   17.01|          22.78|
|         17|    200310|   18.12|          22.34|
|         16|    184968|   19.46|          21.83|
|         15|    184004|   19.11|           19.8|
|         19|    178810|   17.63|          22.86|
|         14|    178026|   19.27|           19.8|
|         13|    165355|   18.42|          19.78|
|         12|    159912|    17.8|          19.74|
|         21|    155910|   18.29|          21.88|
|         20|    155559|   18.05|          22.17|
+-----------+----------+--------+---------------+

Interpretation: 

Evening hours around 3 p.m onwards have the majority of trip volume, this is because of after work commuting.

Tip percentages tend to hover around 20% with the highest being ~2% more at night
